In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [662]:
df = pd.read_csv('power_system_demand.csv')

In [663]:
df.head()

,Timestamp,Demand,TCL_Load,PriceResp_Load,Wind_Gen,Solar_Gen,Conv_Gen,Storage_SOC,Action_Dispatch,Action_TCL,Action_PriceResp,Reward,Grid_Buy_Price,Grid_Sell_Price
0,2025-01-01 00:00:00,306.953167,50.928637,24.622772,90.357993,0.0,210.480202,0.526245,210.480202,45.940437,16.912081,-74.426815,6.5,3.0
1,2025-01-01 00:15:00,319.095143,55.126560,21.487038,68.556957,0.0,234.205579,0.518234,234.205579,47.337750,11.770759,-79.592571,6.5,3.0
2,2025-01-01 00:30:00,300.927295,47.528411,17.517491,90.465401,0.0,200.977379,0.508052,200.977379,42.215825,13.019082,-69.545442,6.5,3.0
3,2025-01-01 00:45:00,305.214395,59.920047,25.675628,99.167507,0.0,200.647141,0.509335,200.647141,57.679634,13.576543,-71.688526,6.5,3.0
4,2025-01-01 01:00:00,289.096321,58.510145,18.164297,85.201429,0.0,209.436511,0.506897,209.436511,55.400260,15.046083,-74.733983,6.5,3.0


## drop irrelevant columns

In [664]:
# columns to drop TCL_Load	PriceResp_Load	Wind_Gen Conv_Gen	Storage_SOC	Action_Dispatch	Action_TCL	Action_PriceResp	Reward
df = df.drop(columns=['TCL_Load', 'PriceResp_Load', 'Wind_Gen', 'Conv_Gen', 'Storage_SOC', 'Action_Dispatch', 'Action_TCL', 'Action_PriceResp', 'Reward'])

In [665]:
df.head()

,Timestamp,Demand,Solar_Gen,Grid_Buy_Price,Grid_Sell_Price
0,2025-01-01 00:00:00,306.953167,0.0,6.5,3.0
1,2025-01-01 00:15:00,319.095143,0.0,6.5,3.0
2,2025-01-01 00:30:00,300.927295,0.0,6.5,3.0
3,2025-01-01 00:45:00,305.214395,0.0,6.5,3.0
4,2025-01-01 01:00:00,289.096321,0.0,6.5,3.0


In [666]:
df['Solar_Gen'] = df['Solar_Gen'].clip(lower=0)

In [667]:
df.head()

,Timestamp,Demand,Solar_Gen,Grid_Buy_Price,Grid_Sell_Price
0,2025-01-01 00:00:00,306.953167,0.0,6.5,3.0
1,2025-01-01 00:15:00,319.095143,0.0,6.5,3.0
2,2025-01-01 00:30:00,300.927295,0.0,6.5,3.0
3,2025-01-01 00:45:00,305.214395,0.0,6.5,3.0
4,2025-01-01 01:00:00,289.096321,0.0,6.5,3.0


In [668]:
df.isnull().sum()

Timestamp          0
Demand             0
Solar_Gen          0
Grid_Buy_Price     0
Grid_Sell_Price    0
dtype: int64

## aggregating the 15 mins data to hourly data

In [669]:
# Aggregate 15-min data to hourly
df1_agg = df.copy()
df1_agg['Timestamp'] = pd.to_datetime(df1_agg['Timestamp'], errors='coerce')
df1_agg['Hour'] = df1_agg['Timestamp'].dt.floor('H')

# Group by hour and aggregate
df = df1_agg.groupby('Hour').agg({
    'Demand': 'sum',
    'Solar_Gen': 'sum',
    'Grid_Buy_Price': 'first',
    'Grid_Sell_Price': 'first'
}).reset_index()

df.rename(columns={'Hour': 'Timestamp'}, inplace=True)

C:\Users\varun\AppData\Local\Temp\ipykernel_32012\300683261.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df1_agg['Hour'] = df1_agg['Timestamp'].dt.floor('H')


In [670]:
df.head()

,Timestamp,Demand,Solar_Gen,Grid_Buy_Price,Grid_Sell_Price
0,2025-01-01 00:00:00,1232.19,0.0,6.5,3.0
1,2025-01-01 01:00:00,1131.66,0.0,6.5,3.0
2,2025-01-01 02:00:00,1257.58,0.0,6.5,3.0
3,2025-01-01 03:00:00,1209.21,0.0,6.5,3.0
4,2025-01-01 04:00:00,1028.25,0.0,6.5,3.0


In [671]:
# round off the demand and solar gen to 2 decimal places
df["Demand"] = df["Demand"].round(2)
df["Solar_Gen"] = df["Solar_Gen"].round(2)


In [672]:
# convert to csv file
df.to_csv('hourly_data.csv')